# DS-Fall - 02 | Train and Evaluate Model

Notebook này load dữ liệu đã xử lý từ `data/processed`, train mô hình DS-Fall multi-task, đánh giá fall detection và direction classification, rồi lưu model/metrics/figures.

Nguyên lý giữ nguyên:
- Input bắt buộc `(100, 6)` theo channel order `[ax, ay, az, gx, gy, gz]`.
- Fall head học trên mọi sample.
- Direction head chỉ học trên fall sample có hướng rõ ràng thông qua `direction_mask`.
- Train/val/test dùng split theo subject đã tạo ở notebook 1.


In [ ]:
# Mount Google Drive nếu chạy trên Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

import sys
from pathlib import Path

# Chỉ cần đổi PROJECT_ROOT nếu repo nằm ở vị trí khác trên Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/ds-fall')
# Debug local nếu chạy ngoài Colab:
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQ = PROJECT_ROOT / 'requirements.txt'
print('PROJECT_ROOT =', PROJECT_ROOT)
if not REQ.exists():
    print('WARNING: requirements.txt not found. Check PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# Cài dependency chỉ khi thiếu package chính. Cell này an toàn để chạy lại.
import importlib
import subprocess
MODULE_CHECKS = [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn'), ('tensorflow', 'tensorflow')]
def _module_ok(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False
missing = [pkg for module, pkg in MODULE_CHECKS if not _module_ok(module)]
if missing and REQ.exists():
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)])

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from IPython.display import display

from src.config import make_config
from src.utils.io import ensure_dir, load_pickle
from src.utils.seed import set_seed
from src.models.ds_fall import build_ds_fall_model
from src.training.dataset import make_tf_dataset
from src.training.train import compile_ds_fall_model
from src.training.callbacks import build_callbacks
from src.training.evaluate import (
    evaluate_direction,
    evaluate_fall_detection,
    evaluate_per_dataset,
    plot_confusion_matrix,
    save_metrics_json,
)
from src.data.visualization import plot_imu_window, plot_training_curves

set_seed(42)
config = make_config(PROJECT_ROOT)
for path in [config.models_dir, config.logs_dir, config.metrics_dir, config.figures_dir / 'training']:
    ensure_dir(path)


In [ ]:
# Kiểm tra TensorFlow và GPU trước khi train.
# Nếu Colab đang dùng GPU, bật memory growth để TensorFlow không chiếm toàn bộ VRAM ngay từ đầu.
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is available and memory growth is enabled.")
    except RuntimeError as e:
        print("GPU setup error:", e)
else:
    print("No GPU detected. Training will run on CPU.")


## 1. Load processed arrays

Load các file đã tạo từ notebook 1. Cell này kiểm tra shape bắt buộc `(N, 100, 6)`, đồng bộ số sample giữa `X`, label arrays và `metadata.csv`, sau đó in summary nhanh để chắc chắn dữ liệu sẵn sàng train.


In [ ]:
processed = config.processed_dir
X = np.load(processed / 'X.npy')
y_fall = np.load(processed / 'y_fall.npy')
y_direction = np.load(processed / 'y_direction.npy')
direction_mask = np.load(processed / 'direction_mask.npy')
metadata = pd.read_csv(processed / 'metadata.csv')
scaler = load_pickle(processed / 'scaler.pkl')

assert X.ndim == 3 and X.shape[1:] == (100, 6), f'Unexpected X shape: {X.shape}'
assert len(X) == len(metadata) == len(y_fall) == len(y_direction) == len(direction_mask)
assert np.isfinite(X).all(), 'Processed X contains NaN or Inf.'

summary_table = pd.DataFrame([
    {'item': 'X shape', 'value': str(tuple(X.shape))},
    {'item': 'sample shape', 'value': '(100, 6)'},
    {'item': 'channel order', 'value': '[ax, ay, az, gx, gy, gz]'},
    {'item': 'NaN count', 'value': int(np.isnan(X).sum())},
    {'item': 'Inf count', 'value': int(np.isinf(X).sum())},
    {'item': 'BITS resampled windows', 'value': int(metadata['resampled'].astype(bool).sum())},
])

print('Processed data summary')
display(summary_table)
print('Scaler fitted in notebook 1')
display(pd.DataFrame({'channel': scaler['channel_names'], 'mean': scaler['mean'], 'std': scaler['std']}))

print('Metadata preview')
display(metadata.head(10))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=metadata, x='split', ax=axes[0], order=['train', 'val', 'test'])
axes[0].set_title('Windows by split')
sns.countplot(data=metadata, x='dataset', ax=axes[1], order=sorted(metadata['dataset'].unique()))
axes[1].set_title('Windows by dataset')
sns.countplot(data=metadata, x='fall_label', ax=axes[2])
axes[2].set_title('Fall label distribution')
for ax in axes:
    ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(config.figures_dir / 'training' / 'loaded_data_summary.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Create train/validation/test datasets

Dùng cột `split` trong metadata, đã được tạo subject-wise ở notebook 1. Direction target `-1` được thay bằng `0` bên trong data helper, nhưng sample weight của direction là `0`, nên các sample không có direction supervision không ảnh hưởng đến direction loss.


In [ ]:
train_mask = metadata['split'].eq('train').to_numpy()
val_mask = metadata['split'].eq('val').to_numpy()
test_mask = metadata['split'].eq('test').to_numpy()

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
yf_train, yf_val, yf_test = y_fall[train_mask], y_fall[val_mask], y_fall[test_mask]
yd_train, yd_val, yd_test = y_direction[train_mask], y_direction[val_mask], y_direction[test_mask]
dm_train, dm_val, dm_test = direction_mask[train_mask], direction_mask[val_mask], direction_mask[test_mask]
meta_test = metadata[test_mask].reset_index(drop=True)

BATCH_SIZE = 64
train_ds = make_tf_dataset(X_train, yf_train, yd_train, dm_train, batch_size=BATCH_SIZE, shuffle=True, seed=42)
val_ds = make_tf_dataset(X_val, yf_val, yd_val, dm_val, batch_size=BATCH_SIZE, shuffle=False)

split_summary = pd.DataFrame([
    {'split': 'train', 'windows': len(X_train), 'fall_windows': int(yf_train.sum()), 'direction_supervised': int(dm_train.sum())},
    {'split': 'val', 'windows': len(X_val), 'fall_windows': int(yf_val.sum()), 'direction_supervised': int(dm_val.sum())},
    {'split': 'test', 'windows': len(X_test), 'fall_windows': int(yf_test.sum()), 'direction_supervised': int(dm_test.sum())},
])
print('Train/val/test arrays')
display(split_summary)
print('tf.data datasets are ready. Batch size =', BATCH_SIZE)


## 3. Build and train DS-Fall

Mô hình giữ đúng nguyên lý DS-Fall:
- tách acc stream và gyro stream,
- fusion sau encoder,
- DS-TCN dilated temporal blocks,
- gated attention pooling,
- hai output heads: `fall_output` và `direction_output`.

Direction loss dùng sample weight từ `direction_mask`, nên chỉ tính loss cho fall sample có hướng rõ ràng.


In [ ]:
# Build model với input cố định 100 x 6.
model = build_ds_fall_model(input_shape=(100, 6), num_direction_classes=3, show_summary=True)

# Compile multi-task loss:
# - fall loss áp dụng cho mọi sample
# - direction loss được mask bằng sample weights
model = compile_ds_fall_model(model, learning_rate=1e-3)
callbacks = build_callbacks(config.models_dir, config.logs_dir, monitor='val_fall_output_accuracy', patience=10)

history = model.fit(train_ds, validation_data=val_ds, epochs=80, callbacks=callbacks)
model.save(config.models_dir / 'ds_fall_final.keras')
print('Saved final model:', config.models_dir / 'ds_fall_final.keras')


## 4. Training curves

Các plot này giúp kiểm tra model có học ổn không: loss tổng, loss từng head, fall accuracy và direction accuracy. File hình được lưu vào `outputs/figures/training/`.


In [ ]:
hist = history.history
train_fig_dir = config.figures_dir / 'training'

plot_training_curves(
    hist,
    ['loss', 'val_loss', 'fall_output_loss', 'val_fall_output_loss', 'direction_output_loss', 'val_direction_output_loss'],
    save_path=train_fig_dir / 'loss_curves.png',
)
plot_training_curves(
    hist,
    ['fall_output_accuracy', 'val_fall_output_accuracy'],
    save_path=train_fig_dir / 'fall_accuracy_curve.png',
)
plot_training_curves(
    hist,
    ['direction_output_accuracy', 'val_direction_output_accuracy'],
    save_path=train_fig_dir / 'direction_accuracy_curve.png',
)

# Compact final-epoch table.
last_epoch = {k: v[-1] for k, v in hist.items() if len(v) > 0}
display(pd.DataFrame([last_epoch]).T.rename(columns={0: 'last_epoch_value'}).round(4))


## 5. Evaluate test set

Đánh giá gồm binary fall detection, direction classification có mask, và per-dataset metrics. BITS được report riêng vì đã qua bước resample 20 Hz -> 50 Hz.


In [ ]:
fall_metrics = evaluate_fall_detection(model, X_test, yf_test)
direction_metrics = evaluate_direction(model, X_test, yd_test, dm_test)
per_dataset_metrics = evaluate_per_dataset(model, X_test, yf_test, yd_test, dm_test, meta_test)

metrics = {
    'fall_detection': fall_metrics,
    'direction_classification': direction_metrics,
    'per_dataset': per_dataset_metrics,
    'bits_preprocessing': '20Hz row-order uniform interpolation to 50Hz',
}
save_metrics_json(config.metrics_dir / 'test_metrics.json', metrics)

fall_summary = pd.DataFrame([{
    'accuracy': fall_metrics.get('accuracy'),
    'precision': fall_metrics.get('precision'),
    'recall': fall_metrics.get('recall'),
    'f1': fall_metrics.get('f1'),
}]).round(4)
print('Fall detection metrics')
display(fall_summary)

if direction_metrics.get('num_supervised', 0) > 0:
    direction_summary = pd.DataFrame([{
        'num_supervised': direction_metrics.get('num_supervised'),
        'accuracy': direction_metrics.get('accuracy'),
        'macro_f1': direction_metrics.get('macro_f1'),
    }]).round(4)
    print('Direction metrics, supervised fall samples only')
    display(direction_summary)
else:
    print('No supervised direction samples in test split.')

per_dataset_rows = []
for dataset_name, result in per_dataset_metrics.items():
    if not isinstance(result, dict) or 'fall' not in result:
        continue
    row = {'dataset': dataset_name}
    row.update({f"fall_{k}": v for k, v in result['fall'].items() if k in ['accuracy', 'precision', 'recall', 'f1']})
    direction_result = result.get('direction', {})
    row['direction_num_supervised'] = direction_result.get('num_supervised', 0)
    row['direction_accuracy'] = direction_result.get('accuracy', np.nan)
    row['direction_macro_f1'] = direction_result.get('macro_f1', np.nan)
    per_dataset_rows.append(row)
print('Per-dataset metrics')
display(pd.DataFrame(per_dataset_rows).round(4))

plot_confusion_matrix(fall_metrics['confusion_matrix'], ['non_fall', 'fall'], 'Fall confusion matrix', save_path=train_fig_dir / 'fall_confusion_matrix.png')
if direction_metrics.get('num_supervised', 0) > 0:
    plot_confusion_matrix(direction_metrics['confusion_matrix'], ['forward', 'backward', 'lateral'], 'Direction confusion matrix', save_path=train_fig_dir / 'direction_confusion_matrix.png')

print('Saved metrics to:', config.metrics_dir / 'test_metrics.json')


## 6. Example predictions

Visualize một vài test windows cùng xác suất dự đoán. Đây là sanity check trực quan: xem tín hiệu 100 x 6 và model confidence cho fall/direction.


In [ ]:
n_examples = min(6, len(X_test))
preds = model.predict(X_test[:n_examples], verbose=0)
fall_probs = preds['fall_output'] if isinstance(preds, dict) else preds[0]
dir_probs = preds['direction_output'] if isinstance(preds, dict) else preds[1]

direction_names = {0: 'forward', 1: 'backward', 2: 'lateral'}
fall_names = {0: 'non_fall', 1: 'fall'}

prediction_rows = []
for i in range(n_examples):
    pred_fall = int(np.argmax(fall_probs[i]))
    pred_dir = int(np.argmax(dir_probs[i]))
    prediction_rows.append({
        'example': i,
        'dataset': meta_test.loc[i, 'dataset'],
        'activity_id': meta_test.loc[i, 'activity_id'],
        'true_fall': fall_names[int(yf_test[i])],
        'pred_fall': fall_names[pred_fall],
        'p_fall': float(fall_probs[i, 1]),
        'true_direction': meta_test.loc[i, 'direction_label'],
        'pred_direction': direction_names[pred_dir],
        'direction_confidence': float(dir_probs[i, pred_dir]),
    })

pred_df = pd.DataFrame(prediction_rows)
print('Example prediction table')
display(pred_df.round(4))

for i in range(min(3, n_examples)):
    title = (
        f"example_{i} | true={pred_df.loc[i, 'true_fall']} | "
        f"pred={pred_df.loc[i, 'pred_fall']} p_fall={pred_df.loc[i, 'p_fall']:.2f} | "
        f"dir={pred_df.loc[i, 'pred_direction']}"
    )
    plot_imu_window(X_test[i], title=title, save_path=train_fig_dir / f'example_prediction_{i}.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=pred_df, x='example', y='p_fall', hue='pred_fall', ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title('Predicted fall probability')
sns.barplot(data=pred_df, x='example', y='direction_confidence', hue='pred_direction', ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title('Predicted direction confidence')
for ax in axes:
    ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(train_fig_dir / 'example_prediction_probabilities.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Optional TFLite export

Xuất bản float32 và dynamic-range quantized TFLite để phục vụ hướng edge deployment. Nếu môi trường không hỗ trợ converter, cell sẽ skip và không ảnh hưởng model `.keras` đã lưu.


In [ ]:
try:
    # TFLite float32.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    (config.models_dir / 'ds_fall_float32.tflite').write_bytes(tflite_model)
    print('Saved float32 TFLite:', config.models_dir / 'ds_fall_float32.tflite')

    # TFLite dynamic range quantization.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quant = converter.convert()
    (config.models_dir / 'ds_fall_dynamic_range.tflite').write_bytes(tflite_quant)
    print('Saved dynamic range quantized TFLite:', config.models_dir / 'ds_fall_dynamic_range.tflite')
except Exception as exc:
    print('TFLite export skipped:', exc)
